## 1. Setup

In [ ]:
!pip install -q scikit-learn matplotlib pillow
import torch
print('GPU:', torch.cuda.is_available())

# ID Forgery Detection: Dual-Stream + LODTO
Run cells top to bottom (Runtime > Change runtime type > GPU).
Set `DATA_SOURCE` in step 2: `demo` (auto-generated, runs instantly) or `folder` (real SIDTD/IDNet data).

## 2. Data source

In [ ]:

DATA_SOURCE = "demo"      # "demo" or "folder"
ROOT = "/content/data"    # used when DATA_SOURCE = "folder": root/<doc_type>/{genuine,forged}/*.jpg
N_PER_CLASS = 120         # demo images per class per document type

## 3. Loader (writes id_forgery_loader.py)

In [ ]:
%%writefile id_forgery_loader.py
"""
Data loader for ID forgery detection (SIDTD / IDNet / any folder of scans).
Works with the dual-stream model: RGB stream + ELA stream + OCR-feature stream.
Includes Leave-One-Document-Type-Out (LODTO) splits.

Expected folder layout (organise your downloaded data like this, or use
build_index_from_paths() to guess labels from file paths):

    root/
      <doc_type_1>/genuine/*.jpg
      <doc_type_1>/forged/*.jpg
      <doc_type_2>/genuine/*.jpg
      ...
"""
import io
import os
import re
import random
from glob import glob

import numpy as np
import torch
from PIL import Image, ImageChops
from torch.utils.data import Dataset, DataLoader

IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")
FORGED_HINTS = ("forged", "fake", "fraud", "tamper", "morph", "replace",
                "portrait", "inpaint", "crop", "mixed", "copy")


# ------------------------------------------------------------------ index
def build_index_from_folders(root):
    """root/<doc_type>/{genuine,forged}/* -> list of (path, label, doc_type). forged=1."""
    items = []
    for doc_type in sorted(os.listdir(root)):
        for label_name, label in (("genuine", 0), ("forged", 1)):
            for p in glob(os.path.join(root, doc_type, label_name, "**", "*"), recursive=True):
                if p.lower().endswith(IMG_EXT):
                    items.append((p, label, doc_type))
    return items


def build_index_from_paths(root, doc_type_depth=1):
    """
    Guess labels from path names (useful for IDNet-style folders where fraud
    types are folders). doc_type = folder name at `doc_type_depth` below root.
    CHECK the printed counts: if genuine/forged look wrong, adjust FORGED_HINTS.
    """
    items = []
    for p in glob(os.path.join(root, "**", "*"), recursive=True):
        if not p.lower().endswith(IMG_EXT):
            continue
        parts = os.path.relpath(p, root).split(os.sep)
        doc_type = parts[min(doc_type_depth - 1, len(parts) - 2)] if len(parts) > 1 else "unknown"
        low = p.lower()
        label = 1 if any(h in low for h in FORGED_HINTS) else 0
        items.append((p, label, doc_type))
    n_f = sum(l for _, l, _ in items)
    print(f"Indexed {len(items)} images | genuine={len(items) - n_f} forged={n_f}")
    return items


# ------------------------------------------------------------------ features
def ela_image(img, quality=90, scale=15):
    """Error Level Analysis: difference between image and its JPEG re-save."""
    buf = io.BytesIO()
    img.save(buf, "JPEG", quality=quality)
    buf.seek(0)
    resaved = Image.open(buf).convert("RGB")
    diff = ImageChops.difference(img, resaved)
    arr = np.clip(np.asarray(diff, dtype=np.float32) * scale, 0, 255)
    return arr / 255.0


# Standard ID formats (India). Extend for other countries.
FORMATS = {
    "aadhaar": re.compile(r"\b\d{4}\s?\d{4}\s?\d{4}\b"),
    "pan": re.compile(r"\b[A-Z]{5}\d{4}[A-Z]\b"),
    "passport": re.compile(r"\b[A-Z]\d{7}\b"),
    "date": re.compile(r"\b\d{2}[/-]\d{2}[/-]\d{4}\b"),
}


def ocr_features(text):
    """Turn OCR text into a fixed vector (8 dims) describing format sanity."""
    if not text:
        return np.zeros(8, dtype=np.float32)
    n = max(len(text), 1)
    feats = [min(len(text) / 500.0, 1.0),
             sum(c.isdigit() for c in text) / n,
             sum(c.isalpha() for c in text) / n,
             sum(c.isupper() for c in text) / n,
             sum(not c.isalnum() and not c.isspace() for c in text) / n]
    feats += [1.0 if FORMATS[k].search(text) else 0.0 for k in ("aadhaar", "pan", "date")]
    return np.array(feats, dtype=np.float32)


def run_ocr(img):
    """Optional OCR. Needs: !apt-get install -y tesseract-ocr && pip install pytesseract"""
    try:
        import pytesseract
        return pytesseract.image_to_string(img)
    except Exception:
        return ""


# ------------------------------------------------------------------ dataset
class IDForgeryDataset(Dataset):
    def __init__(self, items, img_size=224, use_ocr=False, train=False, ocr_cache=None):
        self.items = items
        self.size = img_size
        self.use_ocr = use_ocr
        self.train = train
        self.ocr_cache = ocr_cache if ocr_cache is not None else {}

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        path, label, doc_type = self.items[i]
        img = Image.open(path).convert("RGB").resize((self.size, self.size))
        if self.train and random.random() < 0.5:
            # light brightness jitter only (keep forensic traces intact)
            img = Image.fromarray(np.clip(np.asarray(img) * random.uniform(0.9, 1.1), 0, 255).astype(np.uint8))

        rgb = np.asarray(img, dtype=np.float32) / 255.0
        ela = ela_image(img)
        rgb_t = torch.from_numpy(rgb).permute(2, 0, 1)
        ela_t = torch.from_numpy(ela).permute(2, 0, 1)

        if self.use_ocr:
            if path not in self.ocr_cache:
                self.ocr_cache[path] = ocr_features(run_ocr(img))
            ocr_t = torch.from_numpy(self.ocr_cache[path])
        else:
            ocr_t = torch.zeros(8)

        return rgb_t, ela_t, ocr_t, torch.tensor(label, dtype=torch.float32)


# ------------------------------------------------------------------ LODTO
def lodto_splits(items, max_per_class=None, seed=42):
    """
    Yield (held_out_type, train_items, test_items): train on all other
    document types, test on the held-out one.
    """
    rng = random.Random(seed)
    types = sorted({t for _, _, t in items})

    def cap(lst):
        if max_per_class is None:
            return lst
        out = []
        for lab in (0, 1):
            sub = [x for x in lst if x[1] == lab]
            rng.shuffle(sub)
            out += sub[:max_per_class]
        return out

    for held in types:
        train = cap([x for x in items if x[2] != held])
        test = cap([x for x in items if x[2] == held])
        yield held, train, test


def make_loaders(train_items, test_items, batch_size=32, img_size=224, use_ocr=False, workers=2):
    tr = DataLoader(IDForgeryDataset(train_items, img_size, use_ocr, train=True),
                    batch_size=batch_size, shuffle=True, num_workers=workers)
    te = DataLoader(IDForgeryDataset(test_items, img_size, use_ocr),
                    batch_size=batch_size, shuffle=False, num_workers=workers)
    return tr, te


# ------------------------------------------------------------------ example
if __name__ == "__main__":
    ROOT = "/content/data"                      # change to your dataset folder
    items = build_index_from_folders(ROOT)      # or build_index_from_paths(ROOT)
    print("Doc types:", sorted({t for _, _, t in items}))

    for held, train_items, test_items in lodto_splits(items, max_per_class=300):
        print(f"Held-out: {held} | train={len(train_items)} test={len(test_items)}")
        train_loader, test_loader = make_loaders(train_items, test_items)
        rgb, ela, ocr, y = next(iter(train_loader))
        print(rgb.shape, ela.shape, ocr.shape, y.shape)
        # -> plug train_loader / test_loader into your dual-stream training loop
        break

Writing id_forgery_loader.py


## 4. Build data (demo generator makes 5 doc types with photo swap / text tweak / splice tampering)

In [ ]:
import os, random, numpy as np
from PIL import Image, ImageDraw
from id_forgery_loader import *

DOC_TYPES = ["aadhaar", "passport", "pan", "voter_id", "driving_licence"]
COLORS = {"aadhaar": (255,235,200), "passport": (200,220,255), "pan": (210,240,255),
          "voter_id": (255,220,230), "driving_licence": (225,245,215)}

def make_doc(doc_type, rng):
    img = Image.new("RGB", (320, 200), COLORS[doc_type])
    d = ImageDraw.Draw(img)
    d.rectangle([0, 0, 320, 28], fill=tuple(max(c-90, 0) for c in COLORS[doc_type]))
    d.text((8, 8), doc_type.upper().replace("_", " "), fill=(0, 0, 0))
    d.rectangle([12, 45, 92, 135], fill=tuple(rng.randint(90, 200) for _ in range(3)))   # photo
    num = "".join(str(rng.randint(0, 9)) for _ in range(12))
    d.text((110, 55), "Name: PERSON %d" % rng.randint(100, 999), fill=(0, 0, 0))
    d.text((110, 80), "ID: " + num, fill=(0, 0, 0))
    d.text((110, 105), "DOB: %02d/%02d/%d" % (rng.randint(1,28), rng.randint(1,12), rng.randint(1960,2005)), fill=(0, 0, 0))
    return img

def tamper(img, rng):
    img = img.copy(); d = ImageDraw.Draw(img); kind = rng.choice(["photo", "text", "splice"])
    if kind == "photo":
        patch = Image.effect_noise((80, 90), 40).convert("RGB")
        img.paste(patch, (12, 45))
    elif kind == "text":
        d.rectangle([108, 78, 300, 94], fill=COLORS_BG)
        d.text((110, 80), "ID: " + "".join(str(rng.randint(0,9)) for _ in range(12)), fill=(0,0,0))
    else:
        patch = img.crop((200, 130, 300, 190)).rotate(8)
        img.paste(patch, (rng.randint(100, 200), rng.randint(100, 140)))
    # re-save as JPEG to leave compression traces, like a real edit
    import io; buf = io.BytesIO(); img.save(buf, "JPEG", quality=rng.choice([70, 80, 85])); buf.seek(0)
    return Image.open(buf).convert("RGB")

if DATA_SOURCE == "demo":
    rng = random.Random(0)
    ROOT = "/content/demo_data"
    for t in DOC_TYPES:
        for lab in ("genuine", "forged"):
            os.makedirs(f"{ROOT}/{t}/{lab}", exist_ok=True)
        for i in range(N_PER_CLASS):
            COLORS_BG = COLORS[t]
            g = make_doc(t, rng)
            g.save(f"{ROOT}/{t}/genuine/{i}.jpg", quality=90)
            tamper(make_doc(t, rng), rng).save(f"{ROOT}/{t}/forged/{i}.jpg", quality=90)

items = build_index_from_folders(ROOT)
print(len(items), "images | doc types:", sorted({t for _,_,t in items}))

## 5. Dual-stream model (RGB/noise stream + ELA stream + OCR-format stream)

In [ ]:
import torch.nn as nn

def conv_stream(out=64):
    def blk(i, o): return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))
    return nn.Sequential(blk(3, 16), blk(16, 32), blk(32, 64), nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, out), nn.ReLU())

class DualStream(nn.Module):
    def __init__(self):
        super().__init__()
        self.rgb = conv_stream(); self.ela = conv_stream()
        self.ocr = nn.Sequential(nn.Linear(8, 16), nn.ReLU())
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(64 + 64 + 16, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, rgb, ela, ocr):
        return self.head(torch.cat([self.rgb(rgb), self.ela(ela), self.ocr(ocr)], 1)).squeeze(1)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

## 6. LODTO: train on 4 types, test on the unseen 5th

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve
EPOCHS, IMG, USE_OCR = 4, 128, False   # set USE_OCR=True if you installed tesseract (slower)

def predict(model, loader):
    model.eval(); ys, ps = [], []
    with torch.no_grad():
        for rgb, ela, ocr, y in loader:
            ps += torch.sigmoid(model(rgb.to(device), ela.to(device), ocr.to(device))).cpu().tolist(); ys += y.tolist()
    return np.array(ys), np.array(ps)

results = {}
for held, tr_items, te_items in lodto_splits(items, max_per_class=300):
    tr, te = make_loaders(tr_items, te_items, batch_size=32, img_size=IMG, use_ocr=USE_OCR)
    model = DualStream().to(device)
    opt = torch.optim.Adam(model.parameters(), 1e-3); lossf = nn.BCEWithLogitsLoss()
    for ep in range(EPOCHS):
        model.train()
        for rgb, ela, ocr, y in tr:
            opt.zero_grad(); loss = lossf(model(rgb.to(device), ela.to(device), ocr.to(device)), y.to(device)); loss.backward(); opt.step()
    y, p = predict(model, te)
    results[held] = dict(acc=accuracy_score(y, p > 0.5), auc=roc_auc_score(y, p), fpr_tpr=roc_curve(y, p)[:2])
    print(f"Held-out {held:16s} acc={results[held]['acc']:.3f}  auc={results[held]['auc']:.3f}")

## 7. Charts for the report

In [ ]:
import matplotlib.pyplot as plt
names = list(results)
plt.figure(figsize=(7,4)); plt.bar(names, [results[n]["acc"]*100 for n in names], color="#4C72B0")
plt.ylabel("Accuracy on unseen doc type (%)"); plt.title("LODTO accuracy"); plt.xticks(rotation=20); plt.ylim(0,100)
plt.tight_layout(); plt.savefig("lodto_accuracy.png", dpi=200); plt.show()

plt.figure(figsize=(6,5))
for n in names:
    f, t = results[n]["fpr_tpr"]; plt.plot(f, t, label=f"{n} (AUC={results[n]['auc']:.2f})")
plt.plot([0,1],[0,1],"k--"); plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title("ROC curves (unseen doc type)"); plt.legend()
plt.tight_layout(); plt.savefig("lodto_roc.png", dpi=200); plt.show()

## 8. Live verification (upload any ID image)

In [ ]:
from google.colab import files
final_model = model  # last LODTO model; retrain on all types for deployment

def verify(path):
    ds = IDForgeryDataset([(path, 0, "x")], img_size=IMG, use_ocr=USE_OCR)
    rgb, ela, ocr, _ = ds[0]
    final_model.eval()
    with torch.no_grad():
        p = torch.sigmoid(final_model(rgb[None].to(device), ela[None].to(device), ocr[None].to(device))).item()
    print(f"{path}: {'FORGED' if p > 0.5 else 'GENUINE'}  (forgery score {p:.2f})")

up = files.upload()
for f in up: verify(f)